In [ ]:
!pip install confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 60.2 MB/s eta 0:00:00


In [ ]:
import json
import time
from datetime import datetime
from confluent_kafka import Producer
from google.colab import userdata

# 1. Safely retrieve credentials from Colab Secrets
try:
    kafka_user = userdata.get('KAFKA_USER')
    kafka_password = userdata.get('KAFKA_PASSWORD')
    ca_cert_content = userdata.get('KAFKA_CA')
except userdata.SecretNotFoundError as e:
    print(f"!!! Error: Missing configuration in Colab Secrets (Key icon 🔑). {e}")
    raise

# 2. Write the CA Certificate from secrets into a local file
with open("ca.pem", "w") as f:
    f.write(ca_cert_content.strip())

# 3. Kafka configuration using the safely retrieved variables
conf = {
    'bootstrap.servers': 'kafka-36fd6d61-abhaysharaf-512f.b.aivencloud.com:11895',
    'security.protocol': 'SASL_SSL',
    'sasl.mechanism': 'PLAIN',
    'sasl.username': kafka_user,
    'sasl.password': kafka_password,
    'client.id': 'python-producer',
    'ssl.ca.location': 'ca.pem',
    'ssl.endpoint.identification.algorithm': 'https'
}

producer = Producer(conf)
topic = 'orders'

def delivery_report(err, msg):
    if err is not None:
        print(f"!!! Message delivery failed: {err}")
    else:
        print(f"-> Successfully delivered to {msg.topic()} [Partition: {msg.partition()}]")

print("Starting producer... Sending dummy orders to Aiven:")
try:
    for i in range(1, 20):

        current_timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        order_data = {
            "order_id": 3000 + i,
            "item": f"Laptop Sleeve Type-{i}",
            "qty": i,
            "status": "PLACED",
            "current_time": current_timestamp
        }

        payload = json.dumps(order_data)
        producer.produce(topic, value=payload.encode('utf-8'), callback=delivery_report)
        producer.poll(0)

        print(f"Sent locally to buffer: {payload}")
        time.sleep(1)
except KeyboardInterrupt:
    pass
finally:
    print("Flushing remaining messages out to Aiven Kafka...")
    producer.flush()

Starting producer... Sending dummy orders to Aiven:
Sent locally to buffer: {"order_id": 3001, "item": "Laptop Sleeve Type-1", "qty": 1, "status": "PLACED", "current_time": "2026-06-16 21:33:44"}
-> Successfully delivered to orders [Partition: 0]
Sent locally to buffer: {"order_id": 3002, "item": "Laptop Sleeve Type-2", "qty": 2, "status": "PLACED", "current_time": "2026-06-16 21:33:45"}
-> Successfully delivered to orders [Partition: 0]
Sent locally to buffer: {"order_id": 3003, "item": "Laptop Sleeve Type-3", "qty": 3, "status": "PLACED", "current_time": "2026-06-16 21:33:46"}
-> Successfully delivered to orders [Partition: 0]
Sent locally to buffer: {"order_id": 3004, "item": "Laptop Sleeve Type-4", "qty": 4, "status": "PLACED", "current_time": "2026-06-16 21:33:47"}
-> Successfully delivered to orders [Partition: 0]
Sent locally to buffer: {"order_id": 3005, "item": "Laptop Sleeve Type-5", "qty": 5, "status": "PLACED", "current_time": "2026-06-16 21:33:48"}
-> Successfully delivere